# Mass, dust, redshift, IGM, and AB photometry

This notebook repeats the operations bib applies to a BPASS SED. Each step writes the equation, evaluates it, and checks the result against the function the server calls. The SED here is a smooth stand-in, so the notebook does not need a database. The same functions run on the real grids.

Order of operations, as in `GET /api/v1/photometry` and `GET /api/v1/populations/{id}/sed`:

1. Scale the stored $L_\lambda$ from $10^6\,M_\odot$ formed to the requested mass.
2. Extinguish that rest-frame spectrum (a foreground screen).
3. Redshift: stretch wavelength and convert to observed $f_\lambda$.
4. Optionally multiply the observer-frame spectrum by the mean IGM transmission bluewards of Ly$\alpha(1+z)$.
5. Integrate the observed $f_\lambda$ through the filter for an AB magnitude.

Dust is extinction, not a mixed star–dust attenuation, and the light removed is not re-emitted. Calzetti et al. (2000) published their curve as an attenuation law; bib still multiplies by $10^{-0.4 A_\lambda}$, the same screen as the other curves. Each curve is applied only inside its published wavelength window; outside it the spectrum is left unchanged rather than extrapolated, and photometry through such passbands is flagged (`dust_status`).


In [1]:
import numpy as np
import astropy.units as u
from astropy.cosmology import FlatLambdaCDM
from dust_extinction.parameter_averages import CCM89

from bib.science.dust import apply_dust
from bib.science.photometry import AB_ZERO_FNU, C_A_S, FilterCurve, ab_magnitude
from bib.science.redshift import observe
from bib.science.sed import L_SUN_ERG_S, PC_CM, flux_at_10pc, scale_by_mass

# Stand-in rest-frame SED: L_sun/Å per 1e6 Msun formed, inside the CCM89 window.
wl = np.linspace(1200.0, 25000.0, 4000)
flux = 2.0e4 * (wl / 5500.0) ** -1.2


## Formed mass

BPASS tabulates a population of $10^6\,M_\odot$ formed. A different formed mass is a constant factor on $L_\lambda$:

$$
L_\lambda(M) = L_\lambda(10^6\,M_\odot)\times (M / 10^6\,M_\odot).
$$


In [2]:
mass = 3.2e7  # Msun
scaled = scale_by_mass(flux, mass)
manual_scaled = flux * (mass / 1.0e6)
assert np.allclose(scaled, manual_scaled)
print(f"scale_by_mass matches L_lambda * (M / 1e6) at {mass:.3e} Msun")


scale_by_mass matches L_lambda * (M / 1e6) at 3.200e+07 Msun


## Dust extinction

$$
A_V = R_V\,E(B-V), \qquad
F'_\lambda = F_\lambda\,10^{-0.4 A_\lambda}, \qquad
A_\lambda = (A_\lambda/A_V)\,A_V.
$$

CCM89 and Fitzpatrick (1999) come from `dust_extinction.parameter_averages`. The SMC bar curve is `dust_extinction.averages.G03_SMCBar`. Calzetti et al. (2000) is the polynomial in the next cell, evaluated as $k(\lambda)/R_V$ with $R_V = 4.05$. Outside a curve's wavelength window the server applies **no extinction at all** (factor 1): nothing is extrapolated, and the window is returned with the result so that affected photometry can be flagged. The cell below checks the in-window algebra against `dust_extinction` directly and then demonstrates the no-extrapolation rule.


In [3]:
ebv, rv = 0.15, 3.1
dusted = apply_dust(wl, scaled, "ccm89", ebv, rv)

x = 1.0e4 / wl  # 1/micron; the stand-in SED lies entirely inside the CCM89 window (0.3–10 1/micron)
assert dusted.applied_mask.all()
a_over_av = np.asarray(CCM89(Rv=rv)(x / u.micron), dtype=float)
a_lambda = a_over_av * rv * ebv
manual_dust = scaled * 10.0 ** (-0.4 * a_lambda)
assert np.allclose(dusted.flux, manual_dust)
print(f"CCM89 screen: A_V = {dusted.a_v:.4f}, max |fractional residual| = {np.max(np.abs(dusted.flux / manual_dust - 1)):.1e}")

# Calzetti et al. (2000), valid ~0.12–2.2 µm. k(λ) as published, then A(λ)/A_V = k/R_V.
wl_c = np.linspace(1300.0, 21000.0, 1500)
base = np.full_like(wl_c, 1.0e3)
rv_c, ebv_c = 4.05, 0.2
got = apply_dust(wl_c, base, "calzetti00", ebv_c, None)
xx = 1.0e4 / wl_c
k = np.where(
    xx < 1.0 / 0.63,
    2.659 * (-1.857 + 1.040 * xx) + rv_c,
    2.659 * (-2.156 + 1.509 * xx - 0.198 * xx**2 + 0.011 * xx**3) + rv_c,
)
a_over_c = np.clip(k, 0.0, None) / rv_c
expect = base * 10.0 ** (-0.4 * a_over_c * rv_c * ebv_c)
assert np.allclose(got.flux, expect)
print(f"Calzetti (2000) polynomial matches apply_dust; R_V fixed at {got.rv}")

# No extrapolation: wavelengths outside the window come back untouched.
wl_wide = np.array([500.0, 2000.0, 5500.0, 20000.0, 2.0e5])
out = apply_dust(wl_wide, np.ones(wl_wide.size), "ccm89", ebv, rv)
print("CCM89 window (Å):", [round(v) for v in out.applied_range_a])
print("applied_mask:", out.applied_mask.tolist())
assert out.flux[0] == 1.0 and out.flux[-1] == 1.0 and np.all(out.flux[1:4] < 1.0)


CCM89 screen: A_V = 0.4650, max |fractional residual| = 6.7e-16
Calzetti (2000) polynomial matches apply_dust; R_V fixed at 4.05
CCM89 window (Å): [1000, 33333]
applied_mask: [False, True, True, True, False]


## Redshift

Rest-frame $L_\lambda$ is a luminosity density. At redshift $z$ the observed wavelength and flux density are

$$
\lambda_\mathrm{obs} = \lambda_\mathrm{rest}(1+z), \qquad
f_\lambda(\lambda_\mathrm{obs}) = \frac{L_\lambda(\lambda_\mathrm{rest})\,L_\odot}{4\pi D_L^2\,(1+z)}.
$$

$L_\odot = 3.848\times 10^{26}\,\mathrm{W}$ (the BPASS manual value). $D_L$ is from flat $\Lambda$CDM with $H_0 = 70\,\mathrm{km\,s^{-1}\,Mpc^{-1}}$ and $\Omega_m = 0.3$, computed with `astropy.cosmology.FlatLambdaCDM`. The factor $(1+z)$ is the bandwidth stretch, so the bolometric flux is $L_\mathrm{bol}/(4\pi D_L^2)$. Dust has already been applied in the rest frame. $z = 0$ places the source at 10 pc, which is the absolute-magnitude frame.

The SSP Explorer plots that same spectrum in $L_\odot\,\mathrm{Å}^{-1}$ so it can share an axis with the $z = 0$ curve:

$$
L_\lambda^\mathrm{plot} = L_\lambda^\mathrm{rest}\,\frac{(10\,\mathrm{pc}/D_L)^2}{1+z}.
$$

That is $f_\lambda$ converted back by the 10 pc area, $f_\lambda\,4\pi (10\,\mathrm{pc})^2 / L_\odot$. The browser estimates $D_L$ with a trapezoidal integral of $1/E(z)$. The API uses astropy. The two distances agree to a few parts in $10^6$ over the redshift range of the slider.


In [4]:
z = 0.2
cosmo = FlatLambdaCDM(H0=70.0, Om0=0.3)
wl_obs, flam = observe(wl, dusted.flux, z)
dl_cm = float(cosmo.luminosity_distance(z).to_value("cm"))
flam_manual = dusted.flux * L_SUN_ERG_S / (4.0 * np.pi * dl_cm**2 * (1.0 + z))
assert np.allclose(wl_obs, wl * (1.0 + z))
assert np.allclose(flam, flam_manual)

plot = dusted.flux * (10.0 / (dl_cm / PC_CM)) ** 2 / (1.0 + z)
plot_from_flam = flam * 4.0 * np.pi * (10.0 * PC_CM) ** 2 / L_SUN_ERG_S
assert np.allclose(plot, plot_from_flam)
print(f"z = {z}: API f_lambda matches L_lambda * Lsun / (4 pi D_L^2 (1+z))")
print(f"explorer plot level matches f_lambda * 4 pi (10 pc)^2 / Lsun")


def luminosity_distance_mpc_as_in_the_browser(z, h0=70.0, om0=0.3):
    # Trapezoidal D_L used by sed.js. z > 0.
    ol = 1.0 - om0
    dh = 299792.458 / h0  # Mpc
    n = max(48, int(np.ceil(z * 80)))
    dz = z / n

    def e(zi):
        return np.sqrt(om0 * (1.0 + zi) ** 3 + ol)

    integ = 0.5 / e(0.0)
    for i in range(1, n):
        integ += 1.0 / e(i * dz)
    integ += 0.5 / e(z)
    return (1.0 + z) * dh * integ * dz


print("D_L (Mpc): astropy vs the explorer trapezoid")
for zz in (0.1, 1.0, 2.0, 10.0):
    astropy_dl = float(cosmo.luminosity_distance(zz).to_value("Mpc"))
    browser_dl = luminosity_distance_mpc_as_in_the_browser(zz)
    print(f"  z = {zz:4.1f}   {astropy_dl:12.4f}   {browser_dl:12.4f}   rel {(browser_dl - astropy_dl) / astropy_dl:+.3e}")
    assert abs(browser_dl - astropy_dl) / astropy_dl < 1e-4


z = 0.2: API f_lambda matches L_lambda * Lsun / (4 pi D_L^2 (1+z))
explorer plot level matches f_lambda * 4 pi (10 pc)^2 / Lsun
D_L (Mpc): astropy vs the explorer trapezoid
  z =  0.1       460.2999       460.2999   rel -8.297e-08
  z =  1.0      6607.6576      6607.6710   rel +2.029e-06
  z =  2.0     15539.5862     15539.6356   rel +3.174e-06
  z = 10.0    103842.7459    103843.0178   rel +2.618e-06


## IGM attenuation

For $z>0$ an optional prescription multiplies the **observer-frame** spectrum by the mean transmission of intervening neutral hydrogen,

$$
f_\lambda^{\rm obs}(\lambda_{\rm obs}) \to f_\lambda^{\rm obs}(\lambda_{\rm obs})\,\langle e^{-\tau_{\rm eff}(\lambda_{\rm obs},\,z)}\rangle,
\qquad \lambda_{\rm obs} < 1215.67\,(1+z)\ \text{Å},
$$

with $\tau_{\rm eff}$ the Lyman-series plus Lyman-continuum effective optical depth of Inoue et al. (2014, eqs. 21–29 and Table 2) or Madau (1995, eq. 15 plus the Lyman-continuum approximation). This is a sightline average — no stochastic forest and no damping wing. The cell checks the identity redward of Ly$\alpha(1+z)$, reproduces the Madau (1995) Ly$\alpha$ term by hand, and compares the two prescriptions; `tests/test_igm.py` additionally pins the Inoue et al. (2014) curve to eazy-py's implementation.


In [5]:
from bib.science.igm import LYMAN_ALPHA_A, igm_transmission

z_igm = 3.0
rest_uv = np.linspace(800.0, 1300.0, 501)
obs_uv = rest_uv * (1.0 + z_igm)
t_inoue = igm_transmission(obs_uv, z_igm, "inoue14")
t_madau = igm_transmission(obs_uv, z_igm, "madau95")

# identity redward of Lyα(1+z)
red = obs_uv > LYMAN_ALPHA_A * (1.0 + z_igm)
assert np.all(t_inoue[red] == 1.0) and np.all(t_madau[red] == 1.0)

# Madau (1995) first Lyman-series term by hand, just blueward of Lyα
lobs = 1210.0 * (1.0 + z_igm)
tau_lya = 0.0036 * (lobs / 1215.67) ** 3.46
assert np.isclose(igm_transmission(np.array([lobs]), z_igm, "madau95")[0], np.exp(-tau_lya))

forest = (rest_uv > 1030) & (rest_uv < 1210)
print(f"z = {z_igm}: mean forest transmission  Inoue+14 = {t_inoue[forest].mean():.3f}, Madau95 = {t_madau[forest].mean():.3f}")
lyc = rest_uv < 912
print(f"z = {z_igm}: mean Lyman-continuum transmission  Inoue+14 = {t_inoue[lyc].mean():.3f}, Madau95 = {t_madau[lyc].mean():.3f}")

# applied after redshifting, so it multiplies the observed f_lambda
wl_obs_igm, flam_igm = observe(wl, dusted.flux, z_igm)
flam_igm = flam_igm * igm_transmission(wl_obs_igm, z_igm, "inoue14")
assert np.all(flam_igm[wl_obs_igm > LYMAN_ALPHA_A * (1 + z_igm)] == observe(wl, dusted.flux, z_igm)[1][wl_obs_igm > LYMAN_ALPHA_A * (1 + z_igm)])
print("IGM transmission only alters the observer-frame spectrum blueward of Lyα(1+z)")


z = 3.0: mean forest transmission  Inoue+14 = 0.742, Madau95 = 0.719
z = 3.0: mean Lyman-continuum transmission  Inoue+14 = 0.375, Madau95 = 0.246
IGM transmission only alters the observer-frame spectrum blueward of Lyα(1+z)


## AB photometry

For a photon-counting detector (CCD and NIRCam filters in the SVO set):

$$
m_\mathrm{AB} = -2.5\log_{10}
\frac{\int f_\lambda\,T(\lambda)\,\lambda\,d\lambda}
     {\int f_\lambda^{3631\,\mathrm{Jy}}\,T(\lambda)\,\lambda\,d\lambda}.
$$

The reference spectrum is $f_\nu = 3631\,\mathrm{Jy}$, written as $f_\lambda = 3631\,\mathrm{Jy}\times c/\lambda^2$. Transmission curves are scaled to a peak of 1. The integral is a trapezoidal rule on the SED wavelength grid, with $T(\lambda)$ interpolated onto that grid. The same photon weight is used for every filter. The SVO marks some Johnson–Cousins curves as energy detectors; that tag is stored and is not applied.

At $z = 0$, $f_\lambda$ is $L_\lambda L_\odot / 4\pi (10\,\mathrm{pc})^2$. At $z > 0$, $f_\lambda$ is the observer-frame value from the redshift step, so the magnitude is apparent and includes the K-correction.


In [6]:
band = (wl > 4500.0) & (wl < 6500.0)
filt = FilterCurve(
    "demo.g",
    "demo g",
    wl[band],
    np.exp(-0.5 * ((wl[band] - 5500.0) / 250.0) ** 2),
    detector_type="photon",
)
result = ab_magnitude(wl, flux, filt)  # absolute: flux is still L_sun/Å

inside = (wl >= filt.support[0]) & (wl <= filt.support[1])
w = wl[inside]
f_lambda = flux_at_10pc(flux[inside])
transmission = np.interp(w, filt.wavelength, filt.transmission, left=0.0, right=0.0)
ab_f_lambda = AB_ZERO_FNU * C_A_S / w**2
numerator = np.trapezoid(f_lambda * transmission * w, w)
denominator = np.trapezoid(ab_f_lambda * transmission * w, w)
mag = -2.5 * np.log10(numerator / denominator)
assert abs(result.ab_mag - mag) < 1e-12
print(f"absolute AB = {result.ab_mag:.6f}  (library and the integral agree)")


absolute AB = -15.628599  (library and the integral agree)


## The steps together

Photometry at redshift uses the extinguished, mass-scaled, observer-frame $f_\lambda$. Because every step before the integral is linear in flux, changing the formed mass by a factor $s$ changes the magnitude by $-2.5\log_{10} s$. Dust, redshift and IGM attenuation do not change that factor.


In [7]:
def apparent(formed_mass):
    rest = apply_dust(wl, scale_by_mass(flux, formed_mass), "ccm89", 0.1, 3.1).flux
    observed_wl, observed_f = observe(wl, rest, 0.15)
    return ab_magnitude(observed_wl, observed_f, filt, flux_unit="flam_cgs").ab_mag

m_1e6 = apparent(1.0e6)
m_1e7 = apparent(1.0e7)
expected = -2.5 * np.log10(1.0e7 / 1.0e6)
print(f"m(10^6 Msun) = {m_1e6:.4f}")
print(f"m(10^7 Msun) = {m_1e7:.4f}")
print(f"m(10^6) - m(10^7) = {m_1e6 - m_1e7:.4f} mag")
print(f"a factor of ten in mass changes the magnitude by {-expected:.4f}")
assert abs((m_1e6 - m_1e7) - (-expected)) < 1e-10


m(10^6 Msun) = 23.9769
m(10^7 Msun) = 21.4769
m(10^6) - m(10^7) = 2.5000 mag
a factor of ten in mass changes the magnitude by 2.5000


## External checks

Two comparisons that do not change the equations above.

`synphot` (STScI) is not a dependency of bib. Its AB magnitude is the same photon-weighted integral, sampled on its own wavelength set. On a smooth spectrum the two should agree at the millimagnitude level once they share a wavelength grid. bib uses that same photon weight for every filter, including Johnson $V$.

BPASS ships a `colours` file of Vega magnitudes for $10^6\,M_\odot$ at 10 pc. bib's photometry is AB, through SVO filter curves, which are not the curves BPASS integrated. The comparison converts bib's AB magnitude to Vega with `synphot`'s Vega spectrum through the same SVO curve, $m_\mathrm{Vega} = m_\mathrm{AB} - m_\mathrm{AB}(\mathrm{Vega})$. A residual that is nearly constant in age is a filter or Vega-spectrum difference. A residual that grows with the slope of the SED would mean the integral itself was wrong.


In [8]:
import json
import os
from pathlib import Path
from urllib.request import urlopen

import yaml

root = Path.cwd()
if not (root / "data" / "filters" / "filters.yaml").is_file():
    root = root.parent
from synphot import Observation, SourceSpectrum, SpectralElement
from synphot.models import Empirical1D

# --- synphot on the stand-in SED, photon-counting g band from the cell above ---
flam0 = flux_at_10pc(flux)
bib_g = ab_magnitude(wl, flux, filt).ab_mag
spec = SourceSpectrum(
    Empirical1D,
    points=wl * u.AA,
    lookup_table=flam0 * u.erg / u.s / u.cm**2 / u.AA,
    fill_value=0,
)
band = SpectralElement(
    Empirical1D, points=filt.wavelength * u.AA, lookup_table=filt.transmission
)
syn_g = float(
    Observation(spec, band, force="taper")
    .effstim(u.ABmag, wavelengths=filt.wavelength * u.AA)
    .value
)
print(f"stand-in g band: bib {bib_g:.6f}   synphot {syn_g:.6f}   difference {bib_g - syn_g:+.6f} mag")
assert abs(bib_g - syn_g) < 0.001

# --- published BPASS Vega colours for the same population as the JWST notebook ---
api = os.environ.get("BIB_API", "https://bib.warwick.ac.uk")  # override to point at another bib instance
with urlopen(
    api + "/api/v1/populations?release=2.2.1&imf=imf135_300&z_label=z020&pop_type=bin",
    timeout=120,
) as response:
    rows = json.load(response)
pop = next(row for row in rows if row["alpha_label"] is None and not row["is_synthetic"])
with urlopen(
    api + f"/api/v1/populations/{pop['id']}/timeseries?table=colours", timeout=120
) as response:
    colours = json.load(response)
meta = yaml.safe_load((root / "data" / "filters" / "filters.yaml").read_text())
vega = SourceSpectrum.from_vega()
bands = {
    "sdss.g": "sdss_g",
    "johnson.v": "mag_v",
    "hst.f555w": "f555w",
    "2mass.j": "mag_j",
}
print(f"population {pop['id']}: bib as Vega, minus the BPASS colours file")
print(f"{'log age':>8}  " + "  ".join(f"{name:>10}" for name in bands))
for log_age in (7.0, 8.0, 9.0, 10.0):
    with urlopen(
        api + f"/api/v1/populations/{pop['id']}/sed?log_age={log_age}&resolution=native&mass=1000000",
        timeout=180,
    ) as response:
        sed = json.load(response)
    wave = np.asarray(sed["wavelength"], dtype=float)
    lum = np.asarray(sed["flux"], dtype=float)
    age_index = colours["log_age"].index(log_age)
    deltas = []
    for name, column in bands.items():
        table = np.loadtxt(root / "data" / "filters" / f"{name}.csv", delimiter=",", skiprows=1)
        curve = FilterCurve(
            name, name, table[:, 0], table[:, 1], detector_type=meta[name]["detector_type"]
        )
        bib_ab = ab_magnitude(wave, lum, curve).ab_mag
        element = SpectralElement(
            Empirical1D, points=curve.wavelength * u.AA, lookup_table=curve.transmission
        )
        vega_ab = float(Observation(vega, element, force="taper").effstim(u.ABmag).value)
        published = colours["columns"][column][age_index]
        deltas.append(f"{(bib_ab - vega_ab) - published:+10.3f}")
    print(f"{log_age:8.1f}  " + "  ".join(deltas))


stand-in g band: bib -15.628599   synphot -15.628664   difference +0.000066 mag
population 307: bib as Vega, minus the BPASS colours file
 log age      sdss.g   johnson.v   hst.f555w     2mass.j


     7.0      +0.007      -0.007      +0.009      -0.058


     8.0      +0.001      -0.009      +0.004      -0.055


     9.0      -0.016      -0.029      +0.006      -0.056


    10.0      -0.028      -0.039      +0.008      -0.057
